# Previsão de preço de imóveis

O notebook trabalha apenas com o dataset limpo e usa como entrada do modelo:

- `real_estate_type`
- `municipality`
- `neighbourhood`
- `size`
- `rooms`
- `bathrooms`
- `garage_spaces`
- `has_condominio`

In [274]:
# Se estiver no Colab e precisar instalar as dependências
# !pip install pandas numpy scikit-learn joblib xgboost

import json
import re
import unicodedata
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

## 1. Configurações gerais

In [275]:
RANDOM_STATE = 42
TARGET = "price"

DATASET_PATH = Path("datasets/vendas_grande_vitoria_clean.csv")

if not DATASET_PATH.exists():
    fallback_paths = [
        Path("vendas_grande_vitoria_clean.csv"),
    ]

    for fallback in fallback_paths:
        if fallback.exists():
            DATASET_PATH = fallback
            break

MODEL_OUTPUT_PATH = "modelo_final_imoveis.joblib"
METADATA_OUTPUT_PATH = "metadata_modelo_final_imoveis.json"

print("Dataset usado:", DATASET_PATH)


Dataset usado: datasets\vendas_grande_vitoria_clean.csv


## 2. Carregamento do dataset limpo


In [276]:
df = pd.read_csv(DATASET_PATH)

print("Shape carregado:", df.shape)
display(df.head())
display(df.info())


Shape carregado: (17719, 10)


,id,price,real_estate_type,municipality,neighbourhood,size,rooms,bathrooms,garage_spaces,has_condominio
0,1.510877e+09,265000.0,apartamento,cariacica,alto lage,46.0,2,1,1,1
1,1.355706e+09,480000.0,apartamento,cariacica,alto lage,160.0,3,2,1,1
2,1.355702e+09,480000.0,apartamento,cariacica,alto lage,166.0,3,2,1,1
3,1.514660e+09,399000.0,casa,cariacica,alto lage,200.0,3,2,0,0
4,1.514636e+09,350000.0,casa,cariacica,alto lage,120.0,3,2,1,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17719 entries, 0 to 17718
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17719 non-null  float64
 1   price             17719 non-null  float64
 2   real_estate_type  17719 non-null  object 
 3   municipality      17719 non-null  object 
 4   neighbourhood     17719 non-null  object 
 5   size              17719 non-null  float64
 6   rooms             17719 non-null  int64  
 7   bathrooms         17719 non-null  int64  
 8   garage_spaces     17719 non-null  int64  
 9   has_condominio    17719 non-null  int64  
dtypes: float64(3), int64(4), object(3)
memory usage: 1.4+ MB


None

## 3. Seleção das variáveis

O modelo usa apenas variáveis diretamente relacionadas ao imóvel e à sua localização.

Variáveis categóricas:

- `real_estate_type`: tipo do imóvel;
- `municipality`: município;
- `neighbourhood`: bairro.

Variáveis numéricas:

- `size`: área em m²;
- `rooms`: quartos;
- `bathrooms`: banheiros;
- `garage_spaces`: vagas;
- `has_condominio`: indicador binário para provável existência de condomínio.

`has_condominio` combina valor de condomínio informado, tipo do imóvel e termos do anúncio. Já `iptu` não foi usado porque é pouco confiável na coleta e pode variar conforme forma de divulgação do anúncio.


In [277]:
CATEGORICAL_FEATURES = [
    "real_estate_type",
    "municipality",
    "neighbourhood",
]

NUMERIC_FEATURES = [
    "size",
    "rooms",
    "bathrooms",
    "garage_spaces",
    "has_condominio",
]

RAW_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

# Não utilizados para treinamento
FORBIDDEN_FEATURES = [
    "title",
    "iptu",
    "has_iptu",
    "image_count",
    "source",
    "property_features",
    "condominium_features",
]

for column in RAW_FEATURES + [TARGET]:
    if column not in df.columns:
        raise ValueError(f"Coluna obrigatória ausente no dataset: {column}")

for column in NUMERIC_FEATURES + [TARGET]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in CATEGORICAL_FEATURES:
    df[column] = df[column].fillna("Não informado").astype(str).str.strip()

print("Features usadas:", RAW_FEATURES)
display(df["real_estate_type"].value_counts(dropna=False))
display(df["municipality"].value_counts(dropna=False))


Features usadas: ['real_estate_type', 'municipality', 'neighbourhood', 'size', 'rooms', 'bathrooms', 'garage_spaces', 'has_condominio']


real_estate_type
apartamento    14340
casa            2854
cobertura        525
Name: count, dtype: int64

municipality
guarapari     4683
vila velha    4536
vitoria       4269
serra         3503
cariacica      628
fundao          56
viana           44
Name: count, dtype: int64

## 4. Filtros de segurança

A limpeza principal já foi feita no `clean_data.py`. Aqui ficam apenas filtros mínimos para evitar falhas de treinamento.


In [278]:
before = len(df)

df = df[
    df[TARGET].notna()
    & df["size"].notna()
    & df[TARGET].between(50_000, 10_000_000)
    & df["size"].between(15, 1000)
    & df["real_estate_type"].ne("Não informado")
    & df["municipality"].ne("Não informado")
].copy()

print(f"Registros removidos no filtro: {before - len(df)}")
print("Shape final para modelagem:", df.shape)

display(df[TARGET].describe())
display(df[RAW_FEATURES].head())


Registros removidos no filtro: 0
Shape final para modelagem: (17719, 10)


count    1.771900e+04
mean     1.044948e+06
std      7.769760e+05
min      1.200000e+05
25%      5.590005e+05
50%      8.500000e+05
75%      1.300000e+06
max      1.000000e+07
Name: price, dtype: float64

,real_estate_type,municipality,neighbourhood,size,rooms,bathrooms,garage_spaces,has_condominio
0,apartamento,cariacica,alto lage,46.0,2,1,1,1
1,apartamento,cariacica,alto lage,160.0,3,2,1,1
2,apartamento,cariacica,alto lage,166.0,3,2,1,1
3,casa,cariacica,alto lage,200.0,3,2,0,0
4,casa,cariacica,alto lage,120.0,3,2,1,0


## 5. Métrica principal: erro percentual médio

Uma métrica importante será o erro percentual médio absoluto:

```python
erro_percentual_medio = mean(abs(y_true - y_pred) / y_true)
```

Ela será apresentada em porcentagem. Por exemplo, `0.20` significa que o modelo erra, em média, **20%** do valor real dos imóveis.

Além dela, também é calculado MAE, RMSE, R², erro percentual mediano e a proporção de previsões dentro de 15%, mas essas métricas serão complementares.


In [279]:
def mean_absolute_percentage_error_custom(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = np.maximum(np.abs(y_true), 1)
    return float(np.mean(np.abs(y_true - y_pred) / denominator))


def median_absolute_percentage_error(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = np.maximum(np.abs(y_true), 1)
    return float(np.median(np.abs(y_true - y_pred) / denominator))


def within_percent(y_true: np.ndarray, y_pred: np.ndarray, percent: float) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = np.maximum(np.abs(y_true), 1)
    relative_error = np.abs(y_true - y_pred) / denominator
    return float(np.mean(relative_error <= percent))


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mape = mean_absolute_percentage_error_custom(y_true, y_pred)
    median_ape = median_absolute_percentage_error(y_true, y_pred)

    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "mape": mape,
        "erro_medio_percentual": mape * 100,
        "median_ape": median_ape,
        "erro_mediano_percentual": median_ape * 100,
        "within_10_percent": within_percent(y_true, y_pred, 0.10),
        "within_15_percent": within_percent(y_true, y_pred, 0.15),
        "within_20_percent": within_percent(y_true, y_pred, 0.20),
    }


def mape_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return -mean_absolute_percentage_error_custom(y, pred)


## 6. Separação treino/teste

O dataset é separado em treino e teste com `test_size=0.2`, ou seja:

- 80% para treino;
- 20% para teste.

A separação é estratificada por faixas de preço para manter imóveis baratos, médios e caros em proporções parecidas nos dois conjuntos.


In [280]:
price_bins = pd.qcut(df[TARGET], q=10, duplicates="drop")

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=price_bins,
)

X_train = train_df[RAW_FEATURES]
y_train = train_df[TARGET]

X_test = test_df[RAW_FEATURES]
y_test = test_df[TARGET]

print("Treino:", train_df.shape)
print("Teste:", test_df.shape)


Treino: (14175, 10)
Teste: (3544, 10)


## 7. Pré-processamento

O pré-processamento é feito com `ColumnTransformer`, separando variáveis numéricas e categóricas.

Para modelos baseados em árvores (`Extra Trees`, `XGBoost` e `Hist Gradient Boosting`):

- variáveis numéricas recebem imputação pela mediana;
- variáveis categóricas recebem imputação pela categoria mais frequente;
- categorias são convertidas por `OneHotEncoder`.

Para o `KNN Regressor`:

- além da imputação, as variáveis numéricas recebem `StandardScaler`;
- isso é necessário porque KNN usa distância, e variáveis em escalas maiores, como `size`, poderiam dominar o cálculo.


In [281]:
def make_sparse_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )


def make_dense_tree_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        sparse_threshold=0.0,
    )


def make_dense_scaled_preprocessor() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        sparse_threshold=0.0,
    )


def make_regressor(estimator, *, dense: bool = False, scaled: bool = False) -> TransformedTargetRegressor:
    if scaled:
        preprocessor = make_dense_scaled_preprocessor()
    elif dense:
        preprocessor = make_dense_tree_preprocessor()
    else:
        preprocessor = make_sparse_preprocessor()

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    return TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1,
    )


## 8. Parâmetros dos modelos

In [282]:
models = {
    "Extra Trees": make_regressor(
        ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=4,
            min_samples_leaf=1,
            max_features=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "KNN Regressor": make_regressor(
        KNeighborsRegressor(
            n_neighbors=7,
            weights="distance",
            metric="minkowski",
        ),
        scaled=True,
    ),
    "Hist Gradient Boosting": make_regressor(
        HistGradientBoostingRegressor(
            max_iter=120,
            learning_rate=0.05,
            max_leaf_nodes=39,
            l2_regularization=0.05,
            random_state=RANDOM_STATE,
        ),
        dense=True,
    ),
  "XGBoost": make_regressor(
    XGBRegressor(
        objective="reg:squarederror",
        n_estimators=1000,
        learning_rate=0.07764527689828517,
        max_depth=6,
        min_child_weight=8,
        subsample=0.9571246555570229,
        colsample_bytree=0.5883663457691475,
        random_state=RANDOM_STATE,
        n_jobs=2,
        tree_method="hist",
    )
  )
}


## 9. Treinamento dos modelos base

In [283]:
results = []
trained_models = {}
predictions = {}

for name, model in models.items():
    print(f"Treinando {name}...")

    # Treinamento
    model.fit(X_train, y_train)

    # Predições
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Métricas
    train_metrics = regression_metrics(y_train, train_pred)
    test_metrics = regression_metrics(y_test, test_pred)

    trained_models[name] = model
    predictions[name] = test_pred

    results.append({
        "modelo": name,

        # -------- Treino --------
        "train_mae": train_metrics["mae"],
        "train_rmse": train_metrics["rmse"],
        "train_r2": train_metrics["r2"],
        "train_mape": train_metrics["erro_medio_percentual"],
        "train_within15": train_metrics["within_15_percent"],

        # -------- Teste --------
        **test_metrics
    })

results_df = pd.DataFrame(results).sort_values(
    "erro_medio_percentual",
    ascending=True
)

display(results_df)

display(
    results_df[
        [
            "modelo",
            "train_mape",
            "erro_medio_percentual",
            "train_r2",
            "r2",
            "train_rmse",
            "rmse",
            "train_within15",
            "within_15_percent",
        ]
    ].round(4)
)

Treinando Extra Trees...


Treinando KNN Regressor...
Treinando Hist Gradient Boosting...
Treinando XGBoost...


,modelo,train_mae,train_rmse,train_r2,train_mape,train_within15,mae,rmse,r2,mape,erro_medio_percentual,median_ape,erro_mediano_percentual,within_10_percent,within_15_percent,within_20_percent
0,Extra Trees,89324.066207,171329.122474,0.951262,8.535356,0.817002,181480.273353,354135.161685,0.794134,0.171983,17.198335,0.099374,9.937413,0.501693,0.617099,0.715858
1,KNN Regressor,59209.067553,135281.317690,0.969613,6.425004,0.851005,190309.842402,398448.526897,0.739390,0.179012,17.901170,0.096067,9.606727,0.510722,0.608634,0.690181
3,XGBoost,172893.598776,295853.358674,0.854669,16.529958,0.597390,198077.404438,347612.720764,0.801647,0.188513,18.851320,0.137147,13.714669,0.393341,0.540914,0.664221
2,Hist Gradient Boosting,210748.846266,367889.709327,0.775280,19.901908,0.509418,218305.002450,378566.900524,0.764749,0.208971,20.897080,0.155314,15.531389,0.334368,0.481941,0.606377


,modelo,train_mape,erro_medio_percentual,train_r2,r2,train_rmse,rmse,train_within15,within_15_percent
0,Extra Trees,8.5354,17.1983,0.9513,0.7941,171329.1225,354135.1617,0.8170,0.6171
1,KNN Regressor,6.4250,17.9012,0.9696,0.7394,135281.3177,398448.5269,0.8510,0.6086
3,XGBoost,16.5300,18.8513,0.8547,0.8016,295853.3587,347612.7208,0.5974,0.5409
2,Hist Gradient Boosting,19.9019,20.8971,0.7753,0.7647,367889.7093,378566.9005,0.5094,0.4819


## 10. Seleção do modelo final

In [284]:
# Definir aqui o modelo que será usado nas próximas células.

modelo_name = "XGBoost"

## 11. Comparação final

In [285]:
base_results_for_comparison = results_df.copy()
base_results_for_comparison["best_cv_mape"] = np.nan

comparison_frames = [base_results_for_comparison]
if "tuned_results_df" in globals() and not tuned_results_df.empty:
    comparison_frames.append(tuned_results_df)

all_results = pd.concat(
    comparison_frames,
    ignore_index=True,
    sort=False,
).sort_values("erro_medio_percentual", ascending=True)

available_models = all_results["modelo"].tolist()
if modelo_name not in available_models:
    raise ValueError(
        f"Modelo selecionado não encontrado: {modelo_name}. "
        f"Modelos disponíveis: {available_models}"
    )

best_model_name = modelo_name
best_model_row = all_results.loc[all_results["modelo"].eq(best_model_name)].iloc[0]

comparison_display = all_results.copy()
comparison_display["selecionado"] = comparison_display["modelo"].eq(best_model_name)

display(
    comparison_display[
        [
            "selecionado",
            "modelo",
            "erro_medio_percentual",
            "erro_mediano_percentual",
            "mae",
            "rmse",
            "r2",
            "within_15_percent",
            "best_cv_mape",
        ]
    ].assign(
        erro_medio_percentual=lambda data: data["erro_medio_percentual"].round(2),
        erro_mediano_percentual=lambda data: data["erro_mediano_percentual"].round(2),
        mae=lambda data: data["mae"].round(2),
        rmse=lambda data: data["rmse"].round(2),
        r2=lambda data: data["r2"].round(4),
        within_15_percent=lambda data: data["within_15_percent"].round(4),
        best_cv_mape=lambda data: data["best_cv_mape"].round(2),
    )
)

print("Modelo selecionado pela variável modelo_name:")
print(best_model_name)
print(f"Erro percentual médio: {best_model_row['erro_medio_percentual']:.2f}%")

,selecionado,modelo,erro_medio_percentual,erro_mediano_percentual,mae,rmse,r2,within_15_percent,best_cv_mape
0,False,Extra Trees,17.20,9.94,181480.27,354135.16,0.7941,0.6171,NaN
4,False,Extra Trees Ajustado,17.79,11.82,188681.28,353661.48,0.7947,0.5847,21.72
1,False,KNN Regressor,17.90,9.61,190309.84,398448.53,0.7394,0.6086,NaN
5,False,XGBoost Ajustado,18.48,13.38,194873.83,345434.04,0.8041,0.5533,21.95
2,True,XGBoost,18.85,13.71,198077.40,347612.72,0.8016,0.5409,NaN
3,False,Hist Gradient Boosting,20.90,15.53,218305.00,378566.90,0.7647,0.4819,NaN


Modelo selecionado pela variável modelo_name:
XGBoost
Erro percentual médio: 18.85%


## 12. Recuperação do modelo selecionado

In [286]:
if best_model_name not in trained_models or best_model_name not in predictions:
    raise ValueError(f"Modelo selecionado não encontrado entre os treinados: {best_model_name}")

best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]

best_metrics = regression_metrics(y_test, best_pred)
best_metrics

{'mae': 198077.40443795852,
 'rmse': 347612.7207644165,
 'r2': 0.8016472447864876,
 'mape': 0.1885132013891707,
 'erro_medio_percentual': 18.851320138917067,
 'median_ape': 0.1371466928668019,
 'erro_mediano_percentual': 13.71466928668019,
 'within_10_percent': 0.39334085778781036,
 'within_15_percent': 0.5409142212189616,
 'within_20_percent': 0.6642212189616253}

## 13. Análise de erro

In [287]:
error_analysis = test_df[
    [
        "id",
        "real_estate_type",
        "municipality",
        "neighbourhood",
        "size",
        "rooms",
        "bathrooms",
        "garage_spaces",
        "has_condominio",
        TARGET,
    ]
].copy()

error_analysis["predicted_price"] = best_pred
error_analysis["absolute_error"] = np.abs(error_analysis[TARGET] - error_analysis["predicted_price"])
error_analysis["absolute_percentage_error"] = error_analysis["absolute_error"] / error_analysis[TARGET]
error_analysis["absolute_percentage_error_percent"] = error_analysis["absolute_percentage_error"] * 100
error_analysis["within_15"] = error_analysis["absolute_percentage_error"] <= 0.15

display(error_analysis[["absolute_error", "absolute_percentage_error_percent"]].describe())

city_analysis = (
    error_analysis
    .groupby("municipality")
    .agg(
        quantidade=(TARGET, "count"),
        erro_medio_percentual=("absolute_percentage_error_percent", "mean"),
        erro_mediano_percentual=("absolute_percentage_error_percent", "median"),
        within_15=("within_15", "mean"),
        mae=("absolute_error", "mean"),
        preco_mediano=(TARGET, "median"),
    )
    .sort_values("erro_medio_percentual", ascending=True)
)

display(city_analysis)

type_analysis = (
    error_analysis
    .groupby("real_estate_type")
    .agg(
        quantidade=(TARGET, "count"),
        erro_medio_percentual=("absolute_percentage_error_percent", "mean"),
        erro_mediano_percentual=("absolute_percentage_error_percent", "median"),
        within_15=("within_15", "mean"),
        mae=("absolute_error", "mean"),
        preco_mediano=(TARGET, "median"),
    )
    .sort_values("erro_medio_percentual", ascending=True)
)

display(type_analysis)


,absolute_error,absolute_percentage_error_percent
count,3.544000e+03,3544.000000
mean,1.980774e+05,18.851320
std,2.856974e+05,19.575174
min,3.125000e+00,0.000548
25%,4.425647e+04,6.086397
50%,1.133138e+05,13.714669
75%,2.432592e+05,25.006367
max,4.552901e+06,236.091321


,quantidade,erro_medio_percentual,erro_mediano_percentual,within_15,mae,preco_mediano
municipality,,,,,,
serra,711,13.256460,9.352561,0.666667,90545.930226,435000.0
vitoria,875,17.882387,12.938107,0.569143,232149.309429,1100000.0
vila velha,913,18.899997,13.552318,0.545455,205676.474278,890000.0
guarapari,905,23.126643,17.400955,0.427624,249936.386844,872800.0
cariacica,119,25.117319,17.660078,0.428571,146166.506434,450000.0
fundao,12,25.228163,16.525633,0.416667,183306.635417,667500.0
viana,9,28.848779,20.650889,0.444444,100997.131944,380000.0


,quantidade,erro_medio_percentual,erro_mediano_percentual,within_15,mae,preco_mediano
real_estate_type,,,,,,
apartamento,2866,17.892281,13.183147,0.556874,182488.045986,808565.0
cobertura,88,22.257387,17.782048,0.431818,416881.321023,1685000.0
casa,590,23.001952,15.837100,0.479661,241169.534375,865999.0


### Diagnóstico detalhado dos maiores erros

In [288]:
detailed_errors = error_analysis.copy()

detailed_errors["signed_error"] = detailed_errors["predicted_price"] - detailed_errors[TARGET]
detailed_errors["signed_percentage_error_percent"] = detailed_errors["signed_error"] / detailed_errors[TARGET] * 100
detailed_errors["price_m2"] = detailed_errors[TARGET] / detailed_errors["size"]
detailed_errors["predicted_price_m2"] = detailed_errors["predicted_price"] / detailed_errors["size"]

bins = [0, 300_000, 500_000, 750_000, 1_000_000, 1_500_000, 2_500_000, np.inf]
labels = ["ate_300k", "300k_500k", "500k_750k", "750k_1m", "1m_1_5m", "1_5m_2_5m", "acima_2_5m"]
detailed_errors["price_band"] = pd.cut(detailed_errors[TARGET], bins=bins, labels=labels, include_lowest=True)

cols_to_inspect = [
    "id",
    "real_estate_type",
    "municipality",
    "neighbourhood",
    "size",
    "rooms",
    "bathrooms",
    "garage_spaces",
    "has_condominio",
    TARGET,
    "predicted_price",
    "absolute_error",
    "absolute_percentage_error_percent",
    "signed_percentage_error_percent",
    "price_m2",
    "predicted_price_m2",
]

def format_for_display(dataframe: pd.DataFrame) -> pd.DataFrame:
    formatted = dataframe.copy()
    money_columns = [TARGET, "predicted_price", "absolute_error", "price_m2", "predicted_price_m2", "preco_mediano", "preco_m2_mediano"]
    percent_columns = ["absolute_percentage_error_percent", "signed_percentage_error_percent", "erro_medio_percentual", "erro_mediano_percentual", "vies_medio_percentual"]
    ratio_columns = ["within_15"]

    for column in money_columns:
        if column in formatted.columns:
            formatted[column] = formatted[column].round(0)

    for column in percent_columns:
        if column in formatted.columns:
            formatted[column] = formatted[column].round(2)

    for column in ratio_columns:
        if column in formatted.columns:
            formatted[column] = (formatted[column] * 100).round(2)

    return formatted

print("20 maiores erros percentuais")
display(
    format_for_display(
        detailed_errors[cols_to_inspect]
        .sort_values("absolute_percentage_error_percent", ascending=False)
        .head(20)
    )
)

print("Bairros com maior erro medio, considerando pelo menos 5 exemplos no teste")
neighbourhood_error = (
    detailed_errors
    .groupby(["municipality", "neighbourhood", "real_estate_type"])
    .agg(
        quantidade=(TARGET, "count"),
        erro_medio_percentual=("absolute_percentage_error_percent", "mean"),
        erro_mediano_percentual=("absolute_percentage_error_percent", "median"),
        vies_medio_percentual=("signed_percentage_error_percent", "mean"),
        preco_mediano=(TARGET, "median"),
        preco_m2_mediano=("price_m2", "median"),
        within_15=("within_15", "mean"),
    )
    .query("quantidade >= 5")
    .sort_values("erro_medio_percentual", ascending=False)
)

display(
    format_for_display(neighbourhood_error.head(25))
)

print("Erro por faixa de preco")
price_band_error = (
    detailed_errors
    .groupby("price_band", observed=False)
    .agg(
        quantidade=(TARGET, "count"),
        erro_medio_percentual=("absolute_percentage_error_percent", "mean"),
        erro_mediano_percentual=("absolute_percentage_error_percent", "median"),
        vies_medio_percentual=("signed_percentage_error_percent", "mean"),
        preco_mediano=(TARGET, "median"),
    )
)

display(
    format_for_display(price_band_error)
)

print("Resumo do vies: negativo = modelo subestimou; positivo = modelo superestimou")
display(
    detailed_errors
    .assign(
        direcao=np.where(detailed_errors["signed_error"] >= 0, "superestimou", "subestimou")
    )
    .groupby(["real_estate_type", "direcao"])
    .agg(
        quantidade=(TARGET, "count"),
        erro_medio_percentual=("absolute_percentage_error_percent", "mean"),
        vies_medio_percentual=("signed_percentage_error_percent", "mean"),
    )
    .sort_values(["real_estate_type", "direcao"])
    .pipe(format_for_display)
)

diagnostic_output_path = Path("datasets/diagnostico_maiores_erros.csv")
detailed_errors.sort_values("absolute_percentage_error_percent", ascending=False).to_csv(
    diagnostic_output_path,
    index=False,
    encoding="utf-8-sig",
)
print(f"Diagnostico salvo em: {diagnostic_output_path}")


20 maiores erros percentuais


,id,real_estate_type,municipality,neighbourhood,size,rooms,bathrooms,garage_spaces,has_condominio,price,predicted_price,absolute_error,absolute_percentage_error_percent,signed_percentage_error_percent,price_m2,predicted_price_m2
13948,1.491267e+09,apartamento,vitoria,do moscoso,70.0,2,2,0,1,175000.0,588160.0,413160.0,236.09,236.09,2500.0,8402.0
13628,1.511287e+09,apartamento,vitoria,bento ferreira,132.0,3,2,2,1,450000.0,1433457.0,983457.0,218.55,218.55,3409.0,10860.0
2725,2.812884e+09,apartamento,guarapari,nova guarapari,200.0,3,5,2,1,790000.0,2512403.0,1722403.0,218.03,218.03,3950.0,12562.0
3571,1.491614e+09,apartamento,guarapari,praia do morro,82.0,3,2,2,1,420000.0,1087762.0,667762.0,158.99,158.99,5122.0,13265.0
685,1.510624e+09,casa,guarapari,adalberto simao nader,90.0,3,2,0,0,220000.0,548772.0,328772.0,149.44,149.44,2444.0,6097.0
2347,1.449629e+09,cobertura,guarapari,enseada azul,178.0,3,4,3,1,950000.0,2317763.0,1367763.0,143.98,143.98,5337.0,13021.0
16932,1.507732e+09,apartamento,vitoria,nova palestina,71.0,3,2,0,1,260000.0,626837.0,366837.0,141.09,141.09,3662.0,8829.0
9565,1.513591e+09,apartamento,vila velha,itapua,60.0,2,2,0,1,295000.0,708157.0,413157.0,140.05,140.05,4917.0,11803.0
12617,1.509495e+09,apartamento,vila velha,praia de itaparica,40.0,1,1,0,1,240000.0,569465.0,329465.0,137.28,137.28,6000.0,14237.0
13396,1.507486e+09,casa,vila velha,sao conrado,62.0,2,1,1,1,250000.0,585970.0,335970.0,134.39,134.39,4032.0,9451.0


Bairros com maior erro medio, considerando pelo menos 5 exemplos no teste


quantidade  \
municipality neighbourhood                  real_estate_type               
cariacica    campo grande                   casa                      11   
serra        planicie da serra              apartamento                6   
cariacica    jardim america                 casa                       6   
guarapari    recanto da sereia              casa                       6   
vila velha   sao conrado                    apartamento                6   
serra        boulevard lagoa                casa                       6   
guarapari    nova guarapari                 apartamento               16   
             centro                         casa                       5   
             meaipe                         casa                       7   
             ipiranga                       apartamento                7   
vila velha   morada do sol                  casa                       8   
guarapari    muquicaba                      apartamento               11   
             nova guarapari                 casa                       6   
             village do sol                 casa                       6   
             itapebussu                     casa                      18   
serra        parque residencial laranjeiras casa                       8   
guarapari    santa monica                   casa                      10   
vitoria      mata da praia                  casa                       8   
fundao       praia grande                   casa                      10   
serra        sao francisco                  apartamento                6   
vila velha   divino espirito santo          apartamento                6   
guarapari    centro                         cobertura                  9   
             portal                         apartamento                7   
vila velha   interlagos                     casa                       5   
guarapari    centro                         apartamento              209   

                                                              erro_medio_percentual  \
municipality neighbourhood                  real_estate_type                          
cariacica    campo grande                   casa                              40.47   
serra        planicie da serra              apartamento                       40.44   
cariacica    jardim america                 casa                              40.25   
guarapari    recanto da sereia              casa                              36.77   
vila velha   sao conrado                    apartamento                       35.89   
serra        boulevard lagoa                casa                              34.73   
guarapari    nova guarapari                 apartamento                       32.64   
             centro                         casa                              32.41   
             meaipe                         casa                              30.50   
             ipiranga                       apartamento                       29.87   
vila velha   morada do sol                  casa                              29.69   
guarapari    muquicaba                      apartamento                       28.55   
             nova guarapari                 casa                              27.66   
             village do sol                 casa                              27.49   
             itapebussu                     casa                              26.27   
serra        parque residencial laranjeiras casa                              25.59   
guarapari    santa monica                   casa                              24.82   
vitoria      mata da praia                  casa                              23.74   
fundao       praia grande                   casa                              23.10   
serra        sao francisco                  apartamento                       23.09   
vila velha   divino espirito santo          apartamento                       22.96   
guarapar

Erro por faixa de preco


,quantidade,erro_medio_percentual,erro_mediano_percentual,vies_medio_percentual,preco_mediano
price_band,,,,,
ate_300k,227,26.66,13.71,23.98,265000.0
300k_500k,537,21.43,13.46,12.08,400000.0
500k_750k,793,16.74,11.32,9.08,650000.0
750k_1m,693,17.96,13.71,1.98,888000.0
1m_1_5m,670,16.89,13.36,-3.62,1250000.0
1_5m_2_5m,444,18.15,14.92,-8.68,1800000.0
acima_2_5m,180,23.07,20.32,-16.30,3100000.0


Resumo do vies: negativo = modelo subestimou; positivo = modelo superestimou


quantidade  erro_medio_percentual  \
real_estate_type direcao                                           
apartamento      subestimou          1462                  14.84   
                 superestimou        1404                  21.08   
casa             subestimou           293                  17.68   
                 superestimou         297                  28.25   
cobertura        subestimou            51                  17.51   
                 superestimou          37                  28.80   

                               vies_medio_percentual  
real_estate_type direcao                              
apartamento      subestimou                   -14.84  
                 superestimou                  21.08  
casa             subestimou                   -17.68  
                 superestimou                  28.25  
cobertura        subestimou                   -17.51  
                 superestimou                  28.80

Diagnostico salvo em: datasets\diagnostico_maiores_erros.csv


## 14. Importância das variáveis

Modelos de árvore como Extra Trees, XGBoost e Hist Gradient Boosting permitem extrair importância das variáveis. No caso do KNN, essa análise não está disponível diretamente.


In [289]:
def get_feature_names_from_model(model: TransformedTargetRegressor) -> list[str]:
    pipeline = model.regressor_
    preprocessor = pipeline.named_steps["preprocessor"]

    feature_names = []
    feature_names.extend(NUMERIC_FEATURES)

    onehot = preprocessor.named_transformers_["cat"].named_steps["onehot"]
    categorical_names = onehot.get_feature_names_out(CATEGORICAL_FEATURES)
    feature_names.extend(categorical_names)

    return list(feature_names)


try:
    model_inside = best_model.regressor_.named_steps["model"]

    if hasattr(model_inside, "feature_importances_"):
        feature_names = get_feature_names_from_model(best_model)

        importances = pd.DataFrame(
            {
                "feature": feature_names,
                "importance": model_inside.feature_importances_,
            }
        ).sort_values("importance", ascending=False)

        display(importances.head(30))
    else:
        print("O melhor modelo não possui feature_importances_.")
except Exception as exc:
    print("Não foi possível extrair importâncias:", exc)


,feature,importance
11,municipality_serra,0.158344
8,municipality_cariacica,0.058234
14,municipality_vitoria,0.055947
1,rooms,0.041381
3,garage_spaces,0.040918
6,real_estate_type_casa,0.040777
55,neighbourhood_castelandia,0.025697
12,municipality_viana,0.023809
219,neighbourhood_praia de itaparica,0.020355
10,municipality_guarapari,0.020039


## 15. Salvamento do artefato

In [290]:
artifact = {
    "model": best_model,
    "model_name": best_model_name,
    "target": TARGET,
    "raw_features": RAW_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "forbidden_features": FORBIDDEN_FEATURES,
    "primary_metric": "mape",
    "primary_metric_description": "erro percentual médio absoluto",
    "metrics": best_metrics,
}

joblib.dump(artifact, MODEL_OUTPUT_PATH)

metadata = {
    "model_name": best_model_name,
    "target": TARGET,
    "raw_features": RAW_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "forbidden_features": FORBIDDEN_FEATURES,
    "primary_metric": "mape",
    "primary_metric_description": "erro percentual médio absoluto",
    "metrics": best_metrics,
    "all_results": all_results.to_dict(orient="records"),
}

with open(METADATA_OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print(f"Modelo salvo em: {MODEL_OUTPUT_PATH}")
print(f"Metadados salvos em: {METADATA_OUTPUT_PATH}")


Modelo salvo em: modelo_final_imoveis.joblib
Metadados salvos em: metadata_modelo_final_imoveis.json


## 16. Função de preparação para inferência

A função abaixo garante que novas amostras tenham as mesmas colunas usadas no treinamento.


In [291]:
CONDOMINIO_INDICATOR_PATTERN = (
    r"\b(?:condominio|condomínio|cond\.|portaria|porteiro|elevador|predio|prédio|"
    r"edificio|edifício|residencial|lazer|clube|playground|sal[aã]o de festas|"
    r"area de lazer|área de lazer|piscina|churrasqueira)\b"
)


def normalize_label(value: object) -> str:
    if pd.isna(value):
        return "nao informado"

    text = str(value).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"\s+", " ", text).strip()
    return text or "nao informado"


def infer_has_condominio(samples: pd.DataFrame) -> pd.Series:
    if "condominio" in samples.columns:
        condominio = pd.to_numeric(samples["condominio"], errors="coerce").fillna(0)
    else:
        condominio = pd.Series(0, index=samples.index)

    if "real_estate_type" in samples.columns:
        real_estate_type = samples["real_estate_type"].fillna("").astype(str).map(normalize_label)
    else:
        real_estate_type = pd.Series("", index=samples.index)

    text_parts = []
    for column in ["title", "raw_text", "description"]:
        if column in samples.columns:
            text_parts.append(samples[column].fillna("").astype(str))

    if text_parts:
        text = text_parts[0]
        for part in text_parts[1:]:
            text = text + " " + part
    else:
        text = pd.Series("", index=samples.index)

    return (
        (condominio > 0)
        | real_estate_type.isin(["apartamento", "cobertura"])
        | text.str.contains(CONDOMINIO_INDICATOR_PATTERN, case=False, regex=True)
    ).astype(int)


def prepare_new_samples(samples: pd.DataFrame, artifact: dict[str, Any]) -> pd.DataFrame:
    samples = samples.copy()

    if "has_condominio" in artifact["numeric_features"]:
        inferred_has_condominio = infer_has_condominio(samples)

        if "has_condominio" not in samples.columns:
            samples["has_condominio"] = inferred_has_condominio
        else:
            current_has_condominio = pd.to_numeric(samples["has_condominio"], errors="coerce")
            samples["has_condominio"] = current_has_condominio.fillna(inferred_has_condominio)

    for column in artifact["categorical_features"]:
        if column not in samples.columns:
            samples[column] = "nao informado"

        samples[column] = samples[column].fillna("nao informado").map(normalize_label)

    for column in artifact["numeric_features"]:
        if column not in samples.columns:
            samples[column] = 0

        samples[column] = pd.to_numeric(samples[column], errors="coerce")

    return samples[artifact["raw_features"]]


## 17. Teste de inferencia

In [292]:
loaded_artifact = joblib.load(MODEL_OUTPUT_PATH)

new_samples = pd.DataFrame([
    {
        "real_estate_type": "apartamento",
        "municipality": "Vila Velha",
        "neighbourhood": "Praia da Costa",
        "size": 90,
        "rooms": 2,
        "bathrooms": 2,
        "garage_spaces": 2,
        "has_condominio": 1,
    }
])

X_new = prepare_new_samples(new_samples, loaded_artifact)
prediction = loaded_artifact["model"].predict(X_new)

print(f"Preço previsto: R$ {prediction[0]:,.2f}")


Preço previsto: R$ 1,160,318.25
